# 实验 4 进程调度

## 实验目的

理解操作系统的进程调度方法。采用动态优先数，模拟 P1 至 P5 的处理器调度，通过逐步运行观察就绪链表和进程控制块 PCB 的变化。

这份 Notebook 同时包含实验说明、完整程序、上机步骤、测试数据、已保存的运行结果和总结，可作为本实验的单文件提交件。所有程序都在本文件中，不依赖同目录的 Python 文件或其他说明文件。代码的说明均放在前面的 Markdown 单元格中。

本实验只做调度模拟，省略进程现场保护和恢复。一个模拟时间单位表示执行一次“优先数减 1、剩余时间减 1”。

## 运行环境和启动方法

使用 Python 3.10 或以上版本，以及 JupyterLab、ipywidgets。算法本身只使用 Python 标准库。在准备运行 Notebook 的 Python 环境中，打开终端执行：

```bash
python -m pip install "jupyterlab>=4,<5" "ipywidgets>=8.1,<9"
python -m jupyter lab
```

Linux 中如果 Python 命令是 `python3`，将上面两处 `python` 换成 `python3`。在 JupyterLab 中打开本文件，选择 Python 3 内核，运行全部单元格，即可使用第 8 节的参数表和三个按钮。

GitHub 会将 Notebook 显示为静态页面，可以直接阅读代码和本文件保存的测试结果。按钮需要下载本文件后在 Jupyter 中运行，并保持 Python 内核连接。[GitHub Notebook 显示说明](https://docs.github.com/en/repositories/working-with-files/using-files/working-with-non-code-files#working-with-jupyter-notebook-files-on-github)

## 对照实验要求

| 原文要求 | 本 Notebook 的实现 |
|---|---|
| （1）五个进程及 PCB | 固定 P1 至 P5，保留进程名、指针、剩余时间、优先数、R/E 状态 |
| （2）每次运行前确定参数 | 在参数表中输入五组整数，点击初始化 |
| （3）按优先数建立链式队列 | 队首和 next 保存 PCB 的 ID，末尾指针为 0 |
| （4）选队首并动态修改优先数 | 每一步将队首 PCB 的优先数和剩余时间各减 1 |
| （5）重新入队或结束 | 剩余时间大于 0 时有序插入，等于 0 时置 E 并退出 |
| （6）持续调度直到结束 | 运行到结束按钮重复执行单步调度 |
| （7）（8）显示过程、数据和结果 | 给出默认数据，每一步显示选中进程、队列和完整 PCB 表 |
| 写出程序并调试，给出测试数据和实验结果 | 第 1 至 8 节给出完整实现和默认实验，第 9 节补充同优先数检查 |
| 整理上机步骤，总结经验和体会 | 第 10、11 节整理操作步骤及结果分析 |
| 完成实验报告并上交程序 | 说明、报告内容和可执行程序合并在这份 Notebook 中 |

## 1 导入基础工具

`dataclass` 用于定义 PCB，`perf_counter_ns` 用于测量一次 Python 调度操作的耗时。这个耗时与实验中的模拟时间单位分别记录。

In [1]:
from dataclasses import dataclass
from time import perf_counter_ns

## 2 定义 PCB

每个 PCB 对应一个进程，初始状态为 R。`next` 保存下一个 PCB 的整数 ID，0 表示空指针，满足原文的链式结构要求。

`id(pcb)` 在对象存活期间保持不变；在 CPython 中它也是对象地址。`memory` 会保存全部 PCB，供 ID 查找和结束后的结果查看。重新初始化时，ID 可以变化。[Python id 文档](https://docs.python.org/3/library/functions.html#id)

In [2]:
@dataclass(eq=False)
class PCB:
    name: str
    priority: int
    remaining: int
    next: int = 0
    state: str = "R"

## 3 建立链表并执行调度

初始化时建立五个 PCB，并沿 ID 指针按优先数从大到小插入。每一步摘下队首，优先数和剩余时间各减 1，再重新入队或置为 E，满足要求（2）至（6）。

原文未规定同优先数的顺序，这里把新插入的进程放在已有同优先数进程之后。优先数不设下限，运行时间必须为正整数。历史记录保存当时的数值，后续调度不会改写先前的记录。

In [3]:
class Scheduler:
    def __init__(self, settings):
        settings = list(settings)
        if len(settings) != 5:
            raise ValueError("必须提供 P1 至 P5 的五组参数")
        for item in settings:
            if not isinstance(item, (list, tuple)) or len(item) != 2:
                raise ValueError("每组参数必须是优先数和要求运行时间")
            priority, runtime = item
            if type(priority) is not int or type(runtime) is not int:
                raise ValueError("优先数和要求运行时间必须是整数")
            if runtime <= 0:
                raise ValueError("要求运行时间必须大于 0")
        self.memory = {}
        self.head = 0
        self.tick = 0
        self.total_runtime = sum(runtime for priority, runtime in settings)
        for index, (priority, runtime) in enumerate(settings, 1):
            pcb = PCB(f"P{index}", priority, runtime)
            pointer = id(pcb)
            self.memory[pointer] = pcb
            self.insert(pointer)
        self.history = [self.snapshot()]

    def insert(self, pointer):
        pcb = self.memory[pointer]
        pcb.next = 0
        if self.head == 0 or pcb.priority > self.memory[self.head].priority:
            pcb.next = self.head
            self.head = pointer
            return
        current = self.head
        while self.memory[current].next != 0:
            following = self.memory[current].next
            if self.memory[following].priority < pcb.priority:
                break
            current = following
        pcb.next = self.memory[current].next
        self.memory[current].next = pointer

    def queue_ids(self):
        pointers = []
        pointer = self.head
        while pointer != 0:
            if pointer in pointers:
                raise RuntimeError("就绪链表出现循环")
            pointers.append(pointer)
            pointer = self.memory[pointer].next
        return tuple(pointers)

    def snapshot(self, selected=None, before=None, elapsed_ns=0):
        return {
            "tick": self.tick,
            "selected": selected,
            "before": before,
            "head": self.head,
            "queue": self.queue_ids(),
            "pcbs": tuple({"id": pointer, "name": pcb.name,
                           "next": pcb.next, "remaining": pcb.remaining,
                           "priority": pcb.priority, "state": pcb.state}
                          for pointer, pcb in self.memory.items()),
            "elapsed_ns": elapsed_ns,
        }

    def step(self):
        if self.head == 0:
            return None
        started = perf_counter_ns()
        pointer = self.head
        pcb = self.memory[pointer]
        self.head = pcb.next
        pcb.next = 0
        before = (pcb.priority, pcb.remaining)
        pcb.priority -= 1
        pcb.remaining -= 1
        self.tick += 1
        if pcb.remaining == 0:
            pcb.state = "E"
        else:
            self.insert(pointer)
        elapsed_ns = perf_counter_ns() - started
        record = self.snapshot(pcb.name, before, elapsed_ns)
        self.history.append(record)
        return record

    def run_to_end(self):
        while self.head != 0:
            self.step()
        return self.history

## 4 显示队列和 PCB

用 `IPython.display` 显示表格，用 `ipywidgets` 创建交互控件。下面将 ID 显示成十六进制，同时显示进程名，便于核对 head、next 与链表的对应关系，满足要求（7）（8）。

In [4]:
from html import escape

import ipywidgets as widgets
from IPython.display import HTML, display


def address(pointer):
    return hex(pointer) if pointer else "0"


def queue_text(record):
    pcbs = {pcb["id"]: pcb for pcb in record["pcbs"]}
    names = [pcbs[pointer]["name"] for pointer in record["queue"]]
    return " → ".join(names + ["0"])


def state_html(record):
    rows = []
    for pcb in record["pcbs"]:
        selected = pcb["name"] == record["selected"]
        background = "#eaf3ff" if selected else "#ffffff"
        values = (pcb["name"], address(pcb["id"]), address(pcb["next"]),
                  pcb["remaining"], pcb["priority"], pcb["state"])
        columns = "".join(f"<td>{escape(str(value))}</td>" for value in values)
        rows.append(f'<tr style="background:{background}">{columns}</tr>')
    header = "".join(f"<th>{label}</th>" for label in
                     ("进程名", "PCB ID", "next ID", "剩余时间", "优先数", "状态"))
    title = "初始状态" if record["tick"] == 0 else f'第 {record["tick"]} 步：选中 {record["selected"]}'
    change = ""
    if record["selected"] is not None:
        pcb = next(pcb for pcb in record["pcbs"] if pcb["name"] == record["selected"])
        priority, remaining = record["before"]
        change = (f'<p>优先数 {priority} → {pcb["priority"]}；'
                  f'剩余时间 {remaining} → {pcb["remaining"]}；'
                  f'本步调度耗时 {record["elapsed_ns"]:,} ns。</p>')
    return (f'<section><h4>{title}</h4>{change}'
            f'<p>就绪队列：<strong>{queue_text(record)}</strong></p>'
            f'<p>head = <code>{address(record["head"])}</code></p>'
            '<div style="overflow-x:auto"><table class="pcb-table">'
            f'<thead><tr>{header}</tr></thead><tbody>{"".join(rows)}</tbody>'
            '</table></div></section>')


def history_html(history):
    rows = []
    for record in history[1:]:
        pcb = next(pcb for pcb in record["pcbs"] if pcb["name"] == record["selected"])
        priority, remaining = record["before"]
        rows.append(f'<tr><td>{record["tick"]}</td><td>{record["selected"]}</td>'
                    f'<td>{priority} → {pcb["priority"]}</td>'
                    f'<td>{remaining} → {pcb["remaining"]}</td>'
                    f'<td>{queue_text(record)}</td><td><details>'
                    f'<summary>查看 PCB</summary>{state_html(record)}</details></td></tr>')
    if not rows:
        return "<p>尚未调度。</p>"
    return ('<h4>逐步记录</h4><div style="overflow-x:auto"><table class="pcb-table">'
            '<thead><tr><th>步数</th><th>选中进程</th><th>优先数</th>'
            '<th>剩余时间</th><th>运行后的队列</th><th>完整状态</th></tr></thead>'
            f'<tbody>{"".join(rows)}</tbody></table></div>')


def styled_html(content):
    return HTML('<style>.exp4-view{line-height:1.65}.exp4-view .pcb-table{'
                'border-collapse:collapse;margin:8px 0 16px;font-size:14px;}'
                '.exp4-view .pcb-table th,.exp4-view .pcb-table td{'
                'border:1px solid #d9d9d9;padding:7px 10px;text-align:left;}'
                '.exp4-view .pcb-table th{background:#f1f5f9;color:#111;}'
                '.exp4-view summary{cursor:pointer;white-space:nowrap;}'
                '.exp4-view h4{margin:12px 0 6px;}</style>'
                f'<div class="exp4-view">{content}</div>')

## 5 测试数据和初始队列

先给五个进程确定一组优先数和要求运行时间。下面每一行依次对应 P1 至 P5，两个数字依次为优先数、要求运行时间，满足要求（2）（8）。总运行时间为 14 个模拟时间单位。

| 进程 | 优先数 | 要求运行时间 |
|---|---:|---:|
| P1 | 3 | 3 |
| P2 | 5 | 4 |
| P3 | 4 | 2 |
| P4 | 2 | 3 |
| P5 | 1 | 2 |

In [5]:
DEFAULT_INPUT = [(3, 3), (5, 4), (4, 2), (2, 3), (1, 2)]
example = Scheduler(DEFAULT_INPUT)
display(styled_html(state_html(example.history[0])))

进程名,PCB ID,next ID,剩余时间,优先数,状态
P1,0x29b4b20f770,0x29b4b315480,3,3,R
P2,0x29b4b01ac10,0x29b4b01aad0,4,5,R
P3,0x29b4b01aad0,0x29b4b20f770,2,4,R
P4,0x29b4b315480,0x29b4b3155b0,3,2,R
P5,0x29b4b3155b0,0,2,1,R


## 6 运行结果

将这组数据运行到结束。逐步记录显示每次被选中的进程及运行后的队列；展开“查看 PCB”可以核对该步全部指针、优先数、剩余时间和状态。末尾检查五个进程均为 E 且队首为 0，满足要求（5）至（8）。

In [6]:
example.run_to_end()
display(styled_html(history_html(example.history)))
display(styled_html(state_html(example.history[-1])))
assert example.tick == example.total_runtime == 14
assert example.head == 0
assert all(pcb.state == "E" and pcb.remaining == 0 and pcb.next == 0
           for pcb in example.memory.values())
print("调度顺序：", " → ".join(record["selected"] for record in example.history[1:]))
print(f"共 {example.tick} 步，五个进程全部结束，队首为 0。")

进程名,PCB ID,next ID,剩余时间,优先数,状态
P1,0x29b4b20f770,0,0,0,E
P2,0x29b4b01ac10,0,0,1,E
P3,0x29b4b01aad0,0,0,2,E
P4,0x29b4b315480,0,0,-1,E
P5,0x29b4b3155b0,0,0,-1,E


调度顺序： P2 → P3 → P2 → P1 → P3 → P2 → P4 → P1 → P2 → P5 → P4 → P1 → P5 → P4
共 14 步，五个进程全部结束，队首为 0。


## 7 创建可编辑参数表和按钮

每个输入框接收一个整数。“初始化”使用表格中的当前数值重建链表；“单步调度”执行一个模拟时间单位；“运行到结束”保留全部步骤并完成调度。修改输入后要再次点击初始化。非法运行时间会显示原因，不会破坏正在查看的实验。

当前 PCB 表以浅蓝色标出刚被选中的进程。记录中的“查看 PCB”保存各步状态，便于回看。结束后调度按钮停用，可以修改参数并重新初始化。

In [7]:
def build_lab(defaults):
    inputs = []
    grid = [widgets.HTML("<b>进程</b>"), widgets.HTML("<b>优先数</b>"),
            widgets.HTML("<b>要求运行时间</b>")]
    for index, (priority, runtime) in enumerate(defaults, 1):
        priority_input = widgets.IntText(value=priority, layout=widgets.Layout(width="120px"))
        runtime_input = widgets.IntText(value=runtime, layout=widgets.Layout(width="120px"))
        inputs.append((priority_input, runtime_input))
        grid.extend([widgets.Label(f"P{index}"), priority_input, runtime_input])
    initialize_button = widgets.Button(description="初始化", button_style="info")
    step_button = widgets.Button(description="单步调度")
    finish_button = widgets.Button(description="运行到结束", button_style="success")
    status = widgets.HTML()
    output = widgets.HTML()
    state = {"scheduler": None}

    def refresh():
        scheduler = state["scheduler"]
        finished = scheduler.head == 0
        step_button.disabled = finished
        finish_button.disabled = finished
        message = "全部进程已结束" if finished else "可以单步调度或运行到结束"
        status.value = (f'<p>{message}。已执行 {scheduler.tick} / '
                        f'{scheduler.total_runtime} 个模拟时间单位。修改参数后请点击初始化。</p>')
        content = state_html(scheduler.history[-1]) + history_html(scheduler.history)
        output.value = styled_html(content).data

    def initialize(_):
        settings = [(priority.value, runtime.value) for priority, runtime in inputs]
        try:
            scheduler = Scheduler(settings)
        except ValueError as error:
            status.value = f'<p style="color:#b42318">初始化失败：{escape(str(error))}</p>'
            return
        state["scheduler"] = scheduler
        refresh()

    def single_step(_):
        state["scheduler"].step()
        refresh()

    def finish(_):
        state["scheduler"].run_to_end()
        refresh()

    initialize_button.on_click(initialize)
    step_button.on_click(single_step)
    finish_button.on_click(finish)
    parameter_table = widgets.GridBox(grid, layout=widgets.Layout(
        grid_template_columns="70px 130px 160px", grid_gap="5px 10px"))
    view = widgets.VBox([
        widgets.HTML("<h3>五进程动态优先数调度</h3>"),
        parameter_table,
        widgets.HBox([initialize_button, step_button, finish_button]),
        status,
        output,
    ])
    initialize(None)
    return {"view": view, "inputs": inputs, "initialize": initialize_button,
            "step": step_button, "finish": finish_button, "status": status,
            "output": output, "state": state}

## 8 交互实验

运行本单元格显示参数表，然后按“填写参数 → 初始化 → 单步调度或运行到结束”的顺序操作。可以先把五个优先数都改成 0、运行时间都改成 2，观察同优先数的轮流调度和负优先数。

保存的文件保留了第 5、6 节的静态结果。交互按钮需要运行本单元格，并保持 Python 内核连接。

In [8]:
lab = build_lab(DEFAULT_INPUT)
display(lab["view"])

运行本单元格以显示可编辑参数表和三个调度按钮。


## 9 补充测试与调试

将五个进程的初始优先数都设为 0、要求运行时间都设为 2。预期按 P1 至 P5 的顺序运行两轮，共 10 步，最终优先数均为 -2。下面用断言核对调度顺序、结束状态和空指针，补充第 6 节的默认数据检查。

In [9]:
equal_example = Scheduler([(0, 2)] * 5)
equal_example.run_to_end()
equal_order = [record["selected"] for record in equal_example.history[1:]]
assert equal_order == [f"P{index}" for index in range(1, 6)] * 2
assert equal_example.tick == 10 and equal_example.head == 0
assert all(pcb.priority == -2 and pcb.remaining == 0
           and pcb.state == "E" and pcb.next == 0
           for pcb in equal_example.memory.values())
assert all(pointer == id(pcb) for pointer, pcb in equal_example.memory.items())
print("同优先数测试数据：五个进程的优先数均为 0，要求运行时间均为 2。")
print("实际调度顺序：", " → ".join(equal_order))
print("检查通过：共 10 步，最终优先数均为 -2，五个进程全部结束，队首为 0。")

同优先数测试数据：五个进程的优先数均为 0，要求运行时间均为 2。
实际调度顺序： P1 → P2 → P3 → P4 → P5 → P1 → P2 → P3 → P4 → P5
检查通过：共 10 步，最终优先数均为 -2，五个进程全部结束，队首为 0。


## 10 上机步骤

1. 按本文件开头的命令准备运行环境，打开本 Notebook，选择 Python 3 内核。
2. 从上到下运行全部单元格，查看第 5 节的默认数据、第 6 节的 14 步结果及第 9 节的补充测试。
3. 在第 8 节输入 P1 至 P5 的优先数和要求运行时间。优先数为整数，要求运行时间为正整数。
4. 点击“初始化”，检查初始队列是否按优先数从大到小连接，head 是否指向队首。
5. 点击“单步调度”，核对选中进程、两个数值各减 1，以及 next 指针和队列的变化。展开“查看 PCB”可回看先前各步。
6. 点击“运行到结束”，检查所有剩余时间是否为 0、状态是否为 E、队首是否为 0。修改参数后，再次初始化即可重复实验。
7. 保存 Notebook，使当前实验结果随程序一起保存在同一个文件中，作为实验报告和程序提交。

## 11 结果分析与实验体会

默认数据中 P2 的初始优先数为 5，因此第一步选中 P2。运行后其优先数变为 4、剩余时间变为 3。P3 原来的优先数也是 4，按本实验的同优先数规则排在重新插入的 P2 之前，因此第二步选中 P3。

默认实验共运行 14 步，等于五个进程要求运行时间的总和。最后五个进程的剩余时间和 next 均为 0、状态均为 E，就绪队列为空。补充实验中，五个相同优先数的进程轮流运行两轮，说明稳定插入规则可以得到可复现的调度顺序。

每次调度都会改变优先数，因此要先摘下队首，再按新优先数插回链表。若只改变优先数而不调整位置，后续就无法保证选中优先数最大的进程。运行时间减到 0 的进程应保留 PCB 供查看，同时从就绪队列移除，避免重复运行。

使用 ID 与映射表可以在 Python 中表达指针连接，并直接核对 PCB 的地址和 next 字段。模拟步数用于检查进程消耗的时间单位；纳秒耗时用于观察 Python 调度操作的执行时间，两者含义不同。